# Phase 2: Monte Carlo Speech Simulation & Term Probability Engine

Loads the fine-tuned Trump LLM, runs 1,000 simulated speeches for an upcoming event,
extracts n-grams, and produces probability rankings for each tracked term.

**Input**: Fine-tuned LoRA adapter + event context + term list

**Output**: `predictions.json` with probability scores for each Kalshi market term

## 0. Auto-Pipeline Trigger Check

When running on Colab's scheduler, this cell checks for a trigger file
on Google Drive. If no trigger is found and it's a scheduled run, it exits
early. For manual runs, it proceeds normally.

In [ ]:
import os
import json
from datetime import datetime

# Mount Drive first (needed to check for trigger)
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/trump_trading_bot'
TRIGGER_DIR = os.path.join(PROJECT_DIR, 'triggers')
TRIGGER_PATH = os.path.join(TRIGGER_DIR, 'training_trigger.json')
PREDICTIONS_DIR = os.path.join(PROJECT_DIR, 'predictions')
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

# Check for automated trigger
AUTO_TRIGGERED = False
trigger_data = {}

if os.path.exists(TRIGGER_PATH):
    with open(TRIGGER_PATH) as f:
        trigger_data = json.load(f)
    AUTO_TRIGGERED = True
    print(f"✅ Prediction trigger found!")
    print(f"   Type: {trigger_data.get('trigger_type', 'unknown')}")
    print(f"   Triggered at: {trigger_data.get('triggered_at', 'unknown')}")
else:
    import sys
    is_scheduled = os.getenv('COLAB_SCHEDULED_RUN', '') == '1'
    if is_scheduled:
        print("⏭️  No trigger found. Scheduled run — exiting early.")
        sys.exit(0)
    else:
        print("ℹ️  No trigger file — running manually. Proceeding.")

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers sentencepiece protobuf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/trump_trading_bot'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'predictions')
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Load Fine-Tuned Model

In [ ]:
from unsloth import FastLanguageModel
import torch

BASE_MODEL = 'unsloth/Meta-Llama-3.1-8B'
ADAPTER_PATH = os.path.join(MODEL_DIR, 'trump_lora_adapter')
CHECKPOINT_PATH = os.path.join(MODEL_DIR, 'checkpoints', 'checkpoint-57')

# Load base model first
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Apply LoRA adapter — try trump_lora_adapter first, fall back to checkpoint
from peft import PeftModel
try:
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    print(f'Loaded adapter from: {ADAPTER_PATH}')
except Exception as e:
    print(f'Could not load {ADAPTER_PATH}: {e}')
    print(f'Falling back to checkpoint: {CHECKPOINT_PATH}')
    model = PeftModel.from_pretrained(model, CHECKPOINT_PATH)

FastLanguageModel.for_inference(model)
print(f'VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## 2. Load Scenario Context & Current Events

Auto-populated by your local server's `export_scenario_context()`.
Falls back to manual defaults if not available. Gemini enriches
the raw topics into per-scenario hot words.

In [ ]:
import json

# Load auto-generated scenario context from your server
SCENARIO_PATH = os.path.join(DATA_DIR, 'scenario_context.json')
TERM_CONTEXT_PATH = os.path.join(DATA_DIR, 'term_context.json')

scenario_ctx = {}
if os.path.exists(SCENARIO_PATH):
    with open(SCENARIO_PATH) as f:
        scenario_ctx = json.load(f)
    print(f"✅ Loaded scenario_context.json (generated: {scenario_ctx.get('generated_at', '?')})")
else:
    print("⚠️  No scenario_context.json — using defaults. Run export on your local server first.")

# Extract fields with fallbacks
upcoming = scenario_ctx.get('upcoming_event', {})
EVENT_CONTEXT = {
    'event_type': upcoming.get('type', 'rally'),
    'location': upcoming.get('location', 'unknown'),
    'date': upcoming.get('date', 'upcoming'),
    'title': upcoming.get('title', ''),
    'topics': upcoming.get('topics', []),
}

SCENARIO_WEIGHTS = scenario_ctx.get('scenario_weights', {
    'rally': 0.40,
    'press_conference': 0.20,
    'chopper_talk': 0.15,
    'fox_interview': 0.15,
    'social_media': 0.10,
})

raw_news_topics = scenario_ctx.get('raw_news_topics', [])
trending_terms = scenario_ctx.get('trending_terms', [])
recently_mentioned = scenario_ctx.get('recently_mentioned', [])

# Load recency data from term_context.json
term_recency = {}
if os.path.exists(TERM_CONTEXT_PATH):
    with open(TERM_CONTEXT_PATH) as f:
        term_data = json.load(f)
    for t in term_data:
        term_recency[t['normalized']] = {
            'recent_count_30d': t.get('recent_count_30d', 0),
            'days_since_last_mention': t.get('days_since_last_mention'),
            'trend_score': t.get('trend_score', 0),
        }

print(f"\n📅 Upcoming event: {EVENT_CONTEXT['event_type']} at {EVENT_CONTEXT['location']}")
print(f"\n📊 Scenario weights:")
for s, w in SCENARIO_WEIGHTS.items():
    bar = '█' * int(w * 40)
    print(f"  {s:<20} {w:.0%}  {bar}")
print(f"\n📈 Trending terms: {', '.join(trending_terms[:10])}")
print(f"📰 Raw news topics: {len(raw_news_topics)}")

In [ ]:
# Gemini enrichment: one cheap API call to summarize raw news into
# structured topics and per-scenario hot words.
# Uses gemini-2.0-flash-lite (~$0.075/1M tokens). Cost: ~$0.0001 per run.

import urllib.request
import json as _json

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '').strip()
SCENARIO_HOT_WORDS = {}
enriched_topics = []

if GEMINI_API_KEY and raw_news_topics:
    event_type = EVENT_CONTEXT.get('event_type', 'rally')
    location = EVENT_CONTEXT.get('location', 'unknown')

    prompt_text = f"""You are helping predict which phrases Donald Trump will use in an upcoming {event_type} in {location}.

Current news headlines and recent speech topics:
{chr(10).join(f'- {t}' for t in raw_news_topics[:15])}

Recently trending terms he has been using:
{', '.join(trending_terms[:10])}

Return a JSON object with this exact structure:
{{
  "enriched_topics": ["topic 1", "topic 2", "topic 3", "topic 4", "topic 5"],
  "rally_hot_words": ["word1", "word2", "word3", "word4", "word5"],
  "press_conference_hot_words": ["word1", "word2", "word3", "word4", "word5"],
  "fox_interview_hot_words": ["word1", "word2", "word3", "word4", "word5"],
  "chopper_talk_hot_words": ["word1", "word2", "word3", "word4", "word5"],
  "social_media_hot_words": ["word1", "word2", "word3", "word4", "word5"]
}}

Rules:
- enriched_topics: 5 concise current event summaries (under 10 words each) Trump would reference
- hot_words: specific words/phrases Trump would use in THAT format (rally = bombastic, press = defensive, interview = storytelling, chopper = punchy, social = ALL CAPS style)
- Return ONLY valid JSON, no explanation or markdown fences."""

    payload = _json.dumps({
        'contents': [{'parts': [{'text': prompt_text}]}],
        'generationConfig': {'temperature': 0.3, 'maxOutputTokens': 512}
    }).encode()

    req = urllib.request.Request(
        f'https://generativelanguage.googleapis.com/v1beta/models/'
        f'gemini-2.0-flash-lite:generateContent?key={GEMINI_API_KEY}',
        data=payload,
        headers={'Content-Type': 'application/json'},
        method='POST'
    )

    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            result = _json.loads(resp.read())
        raw_text = result['candidates'][0]['content']['parts'][0]['text']
        # Strip markdown fences if present
        raw_text = raw_text.strip().lstrip('```json').lstrip('```').rstrip('```').strip()
        gemini_data = _json.loads(raw_text)

        enriched_topics = gemini_data.get('enriched_topics', [])
        SCENARIO_HOT_WORDS = {
            k.replace('_hot_words', ''): v
            for k, v in gemini_data.items()
            if k.endswith('_hot_words')
        }

        print(f'✅ Gemini enrichment successful')
        print(f'\n📰 Enriched topics:')
        for t in enriched_topics:
            print(f'  • {t}')
        print(f'\n🎯 Per-scenario hot words:')
        for scenario, words in SCENARIO_HOT_WORDS.items():
            print(f'  {scenario}: {", ".join(words)}')

    except Exception as e:
        print(f'⚠️  Gemini enrichment failed ({e}) — using raw topics')
else:
    if not GEMINI_API_KEY:
        print('ℹ️  No Gemini API key — skipping enrichment')
    elif not raw_news_topics:
        print('ℹ️  No raw news topics to enrich — skipping')

# Use enriched topics if available, otherwise fall back to raw
news_topics = enriched_topics if enriched_topics else raw_news_topics[:5]
topic_str = '; '.join(news_topics[:3]) if news_topics else 'the economy and national security'
print(f'\nFinal topic string for prompts: "{topic_str[:80]}..."')

In [ ]:
# Load the term list from your bot (the words Kalshi markets are tracking)
TERMS_PATH = os.path.join(DATA_DIR, 'term_context.json')

tracked_terms = []
if os.path.exists(TERMS_PATH):
    with open(TERMS_PATH) as f:
        term_data = json.load(f)
    tracked_terms = [t['normalized'] for t in term_data]
    print(f'Tracking {len(tracked_terms)} terms from Kalshi markets:')
    for t in tracked_terms[:20]:
        print(f'  - {t}')
    if len(tracked_terms) > 20:
        print(f'  ... and {len(tracked_terms) - 20} more')
else:
    print('No term list found. Upload term_context.json to', DATA_DIR)
    # Fallback: common Trump terms
    tracked_terms = [
        'tariff', 'china', 'border', 'wall', 'fake news', 'tremendous',
        'incredible', 'disaster', 'winning', 'great', 'beautiful',
        'billions', 'millions', 'radical left', 'deep state',
    ]

## 3. Build Multi-Scenario Prompt Templates

5 scenario types with distinct speech patterns. Simulations are
allocated across scenarios based on the upcoming event type.

In [ ]:
import random

location = EVENT_CONTEXT.get('location', 'America')
event_type = EVENT_CONTEXT.get('event_type', 'rally')

# Helper: get hot words for a scenario, or fall back to trending terms
def get_hot(scenario):
    words = SCENARIO_HOT_WORDS.get(scenario, trending_terms[:5])
    return words if words else ['the economy', 'trade', 'border']

# ─── SCENARIO PROMPT TEMPLATES ───────────────────────────────────────────────
# Each scenario has 3-5 prompt variations. The fine-tuned model continues
# generating in Trump's style from these seeds.

SCENARIO_TEMPLATES = {
    'rally': [
        f"Thank you {location}! This is an incredible crowd. You know, "
        f"we have to talk about what's happening with {topic_str}. ",

        f"And I have to tell you, {location}, the fake news media won't "
        f"report this but {news_topics[0] if news_topics else 'things are changing fast'}. "
        f"It's a disgrace. ",

        f"We love {location}. We love the people of {location}. "
        f"And we are going to win so big. Let me tell you about {topic_str}. ",

        f"Hello {location}! What a crowd, what a beautiful crowd. Now, "
        f"I want to talk about {get_hot('rally')[0] if get_hot('rally') else 'winning'}. "
        f"Nobody talks about this. ",

        f"This is a big day. A very big day for {location}. Because we are "
        f"going to talk about {news_topics[-1] if len(news_topics) > 1 else 'making America great again'}. ",
    ],

    'press_conference': [
        f"Reporter question about {topic_str}. "
        f"Look, it's very simple. We've been saying this for a long time. ",

        f"Next question. Yes, you. "
        f"About {news_topics[0] if news_topics else 'trade policy'} — "
        f"I'll tell you, nobody knows this better than me. ",

        f"I've already answered that. The answer is very clear. "
        f"On {topic_str}, we are doing better than any administration in history. ",

        f"That's a very unfair question. But I'll answer it anyway. "
        f"On the issue of {get_hot('press_conference')[0] if get_hot('press_conference') else 'policy'}, "
        f"we have done more in two years than others did in eight. ",

        f"Go ahead. What's your question. "
        f"Okay, so on {news_topics[1] if len(news_topics) > 1 else 'that issue'}, "
        f"let me be very clear about something. ",
    ],

    'chopper_talk': [
        f"Shouting over helicopter noise. "
        f"On {news_topics[0] if news_topics else 'tariffs'} — it's very simple. "
        f"We're doing great. ",

        f"Look, I'll tell you this — {topic_str}. "
        f"Very simple. Nobody else will say it. I'll say it. ",

        f"We're heading out now. But I want to say — "
        f"{news_topics[0] if news_topics else 'what they are doing is a disgrace'}. "
        f"Total disgrace. ",

        f"Yeah, I saw that. On {get_hot('chopper_talk')[0] if get_hot('chopper_talk') else 'the news'} — "
        f"it's a witch hunt. Total witch hunt. But we're winning. ",
    ],

    'fox_interview': [
        f"Sean, I have to tell you — and nobody wants to talk about this — "
        f"but {topic_str}. The numbers are incredible. ",

        f"Well first of all, thank you for having me. And look, "
        f"on {news_topics[0] if news_topics else 'trade'}, "
        f"I was right from the very beginning. Everyone said I was wrong. ",

        f"And I said it at the time — I said it years ago — "
        f"{topic_str}. And now look what happened. ",

        f"Laura, the thing people don't understand is "
        f"{get_hot('fox_interview')[0] if get_hot('fox_interview') else 'how well we are doing'}. "
        f"I mean, the numbers speak for themselves. ",
    ],

    'social_media': [
        f"BREAKING: {(news_topics[0] if news_topics else 'GREAT NEWS FOR AMERICA').upper()}!!! "
        f"The FAKE NEWS MEDIA won't tell you this but ",

        f"Crooked Democrats are doing it again. "
        f"{topic_str.upper()}. TOTAL DISASTER! We need to ",

        f"Just spoke with world leaders about {topic_str}. "
        f"AMERICA FIRST always. We are WINNING like never before! ",

        f"The {get_hot('social_media')[0].upper() if get_hot('social_media') else 'RADICAL LEFT'} "
        f"is destroying our Country. But not for long! We will ",
    ],
}

# ─── ALLOCATE SIMULATIONS ACROSS SCENARIOS ────────────────────────────────────

NUM_SIMULATIONS = 1000
scenario_counts = {}
remaining = NUM_SIMULATIONS
scenarios_list = list(SCENARIO_WEIGHTS.keys())

for i, scenario in enumerate(scenarios_list):
    if i == len(scenarios_list) - 1:
        scenario_counts[scenario] = remaining
    else:
        count = round(NUM_SIMULATIONS * SCENARIO_WEIGHTS[scenario])
        scenario_counts[scenario] = count
        remaining -= count

# Build master prompt list: (prompt_text, scenario_name)
PROMPT_SEEDS = []
for scenario, count in scenario_counts.items():
    templates = SCENARIO_TEMPLATES[scenario]
    for i in range(count):
        template = templates[i % len(templates)]
        PROMPT_SEEDS.append((template, scenario))

random.shuffle(PROMPT_SEEDS)

print(f'Simulation budget ({NUM_SIMULATIONS} total):')
for scenario, count in scenario_counts.items():
    bar = '█' * (count // 20)
    print(f'  {scenario:<20} {count:>4} sims  {bar}')
print(f'\nTotal prompts ready: {len(PROMPT_SEEDS)}')
print(f'Sample: [{PROMPT_SEEDS[0][1]}] {PROMPT_SEEDS[0][0][:80]}...')

## 4. Monte Carlo Simulation: Generate 1,000 Speeches

In [ ]:
from tqdm.notebook import tqdm
import random

MAX_NEW_TOKENS = 512       # ~400 words per simulation
TEMPERATURE = 0.75         # slight variation while keeping core patterns
TOP_P = 0.9                # nucleus sampling
BATCH_SIZE = 10            # generate 10 at a time for GPU efficiency

all_speeches = []  # list of {"text": str, "scenario": str}

print(f'Generating {NUM_SIMULATIONS} simulated speeches across {len(scenario_counts)} scenarios...')
print(f'Temperature: {TEMPERATURE} | Max tokens: {MAX_NEW_TOKENS}')
print(f'Estimated time: ~{NUM_SIMULATIONS * 2 / 60:.0f} minutes on A100\n')

for batch_start in tqdm(range(0, NUM_SIMULATIONS, BATCH_SIZE)):
    batch_end = min(batch_start + BATCH_SIZE, NUM_SIMULATIONS)
    batch_items = PROMPT_SEEDS[batch_start:batch_end]

    batch_prompts = [item[0] for item in batch_items]
    batch_scenarios = [item[1] for item in batch_items]

    # Tokenize batch
    inputs = tokenizer(
        batch_prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512,
    ).to('cuda')

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            repetition_penalty=1.1,
        )

    # Decode and tag with scenario
    for j, output in enumerate(outputs):
        text = tokenizer.decode(output, skip_special_tokens=True)
        all_speeches.append({
            'text': text,
            'scenario': batch_scenarios[j],
        })

# Verify scenario distribution
from collections import Counter
actual_counts = Counter(s['scenario'] for s in all_speeches)
print(f'\nGenerated {len(all_speeches)} speeches')
print(f'Average length: {sum(len(s["text"].split()) for s in all_speeches) / len(all_speeches):.0f} words')
print(f'\nActual scenario distribution:')
for scenario, count in sorted(actual_counts.items(), key=lambda x: -x[1]):
    print(f'  {scenario:<20} {count:>4} speeches')

In [ ]:
# Preview samples from each scenario + export all speeches

# Show one sample per scenario
seen_scenarios = set()
for speech in all_speeches:
    s = speech['scenario']
    if s not in seen_scenarios:
        seen_scenarios.add(s)
        print(f'\n{"="*60}')
        print(f'[{s.upper()}]')
        print(f'{"="*60}')
        print(speech['text'][:600])
        print('...\n')
    if len(seen_scenarios) >= 5:
        break

# Export all simulated speeches to Drive for inspection
speeches_export = {
    'generated_at': datetime.utcnow().isoformat() if 'datetime' in dir() else 'unknown',
    'total_speeches': len(all_speeches),
    'scenario_counts': dict(Counter(s['scenario'] for s in all_speeches)),
    'speeches': [
        {
            'index': i,
            'scenario': s['scenario'],
            'word_count': len(s['text'].split()),
            'text': s['text'],
        }
        for i, s in enumerate(all_speeches)
    ],
}

speeches_path = os.path.join(OUTPUT_DIR, 'simulated_speeches.json')
with open(speeches_path, 'w') as f:
    json.dump(speeches_export, f, indent=2)

print(f'\n✅ All {len(all_speeches)} speeches exported to:')
print(f'   {speeches_path}')
print(f'   File size: {os.path.getsize(speeches_path) / 1e6:.1f} MB')

## 5. N-Gram Extraction & Term Probability Ranking

In [ ]:
import re
from collections import Counter

def extract_ngrams(text: str, n_range=(2, 5)) -> list[str]:
    """Extract n-grams from text."""
    words = re.findall(r'\b\w+\b', text.lower())
    ngrams = []
    for n in range(n_range[0], n_range[1] + 1):
        for i in range(len(words) - n + 1):
            ngrams.append(' '.join(words[i:i + n]))
    return ngrams

def check_term_in_speech(speech: str, term: str) -> bool:
    """Check if a term (word or phrase) appears in a speech."""
    pattern = r'\b' + re.escape(term.lower()) + r'\b'
    return bool(re.search(pattern, speech.lower()))

def count_term_in_speech(speech: str, term: str) -> int:
    """Count occurrences of a term in a speech."""
    pattern = r'\b' + re.escape(term.lower()) + r'\b'
    return len(re.findall(pattern, speech.lower()))

print('Extraction functions ready.')

In [ ]:
import math
from collections import defaultdict

# Calculate probability for each tracked Kalshi term
# with recency weighting and per-scenario breakdown

term_results = {}
total_speeches = len(all_speeches)

# Group speeches by scenario for per-scenario stats
speeches_by_scenario = defaultdict(list)
for speech in all_speeches:
    speeches_by_scenario[speech['scenario']].append(speech['text'])

# Recency weight function: terms mentioned recently get a boost,
# stale terms get penalized.  Uses exponential decay with 60-day half-life.
HALF_LIFE_DAYS = 60

def recency_weight(term_name):
    """Return a multiplier [0.2, 1.5] based on how recently a term was used."""
    info = term_recency.get(term_name, {})
    days_since = info.get('days_since_last_mention')
    recent_30d = info.get('recent_count_30d', 0)

    if days_since is None:
        return 1.0  # no data, neutral

    decay = math.exp(-math.log(2) * days_since / HALF_LIFE_DAYS)
    recency_boost = min(0.3, recent_30d * 0.05)
    return max(0.2, min(1.5, decay + recency_boost))

print(f'Analyzing {len(tracked_terms)} tracked terms across {total_speeches} simulated speeches...')
print(f'Scenarios: {", ".join(speeches_by_scenario.keys())}')
print(f'Recency weighting: enabled (half-life={HALF_LIFE_DAYS} days)\n')

for term in tqdm(tracked_terms):
    sub_terms = [t.strip() for t in term.split('/')] if '/' in term else [term]

    # Aggregate stats
    speeches_containing = 0
    total_mentions = 0

    # Per-scenario stats
    by_scenario = {}

    for speech in all_speeches:
        text = speech['text']
        found = False
        mention_count = 0
        for st in sub_terms:
            count = count_term_in_speech(text, st)
            if count > 0:
                found = True
                mention_count += count
        if found:
            speeches_containing += 1
        total_mentions += mention_count

    # Per-scenario breakdown
    for scenario, texts in speeches_by_scenario.items():
        sc_containing = 0
        sc_mentions = 0
        for text in texts:
            found = False
            for st in sub_terms:
                count = count_term_in_speech(text, st)
                if count > 0:
                    found = True
                    sc_mentions += count
            if found:
                sc_containing += 1
        sc_total = len(texts)
        by_scenario[scenario] = {
            'probability': round(sc_containing / sc_total, 4) if sc_total > 0 else 0,
            'speeches_containing': sc_containing,
            'total_in_scenario': sc_total,
            'mentions': sc_mentions,
        }

    raw_probability = speeches_containing / total_speeches
    avg_mentions = total_mentions / total_speeches

    # Apply recency weight
    rw = recency_weight(term)
    adjusted_probability = min(1.0, raw_probability * rw)

    term_results[term] = {
        'probability': round(adjusted_probability, 4),
        'raw_probability': round(raw_probability, 4),
        'recency_weight': round(rw, 3),
        'speeches_containing': speeches_containing,
        'total_mentions': total_mentions,
        'avg_mentions_per_speech': round(avg_mentions, 2),
        'confidence': round(min(1.0, total_speeches / 1000), 2),
        'by_scenario': by_scenario,
    }

# Sort by adjusted probability
sorted_terms = sorted(term_results.items(), key=lambda x: x[1]['probability'], reverse=True)

print(f'\n{"="*100}')
print(f'{"TERM PROBABILITY RANKINGS (recency-weighted, multi-scenario)":^100}')
print(f'{"="*100}')

# Header
scenario_names = list(speeches_by_scenario.keys())
header = f'{"Term":<25} {"Adj":>6} {"Raw":>6} {"Rw":>5}'
for sn in scenario_names:
    header += f' {sn[:8]:>8}'
print(header)
print('-' * 100)

for term, data in sorted_terms:
    adj = f"{data['probability']:.0%}"
    raw = f"{data['raw_probability']:.0%}"
    rw = f"{data['recency_weight']:.2f}"
    row = f'{term:<25} {adj:>6} {raw:>6} {rw:>5}'
    for sn in scenario_names:
        sc_prob = data['by_scenario'].get(sn, {}).get('probability', 0)
        row += f' {sc_prob:.0%}'.rjust(9)
    print(row)

In [ ]:
# Also find the MOST COMMON N-GRAMS across all simulations
# (discovers phrases not in the Kalshi term list that Trump might say)

print('Extracting all n-grams from simulated speeches...')

all_ngrams = Counter()
for speech in tqdm(all_speeches):
    ngrams = extract_ngrams(speech['text'], n_range=(2, 5))
    all_ngrams.update(ngrams)

# Filter out boring n-grams
stopwords = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'to', 'of',
             'in', 'for', 'on', 'with', 'at', 'by', 'and', 'or', 'but', 'not',
             'that', 'this', 'it', 'we', 'they', 'you', 'he', 'she', 'i',
             'do', 'did', 'does', 'have', 'has', 'had', 'will', 'would'}

interesting_ngrams = {}
for ngram, count in all_ngrams.most_common(500):
    words = ngram.split()
    # Skip if all words are stopwords
    if all(w in stopwords for w in words):
        continue
    # Must appear in at least 5% of speeches
    if count / total_speeches >= 0.05:
        interesting_ngrams[ngram] = {
            'count': count,
            'probability': round(count / total_speeches, 4),
        }

print(f'\n{"="*60}')
print(f'{"TOP PREDICTED PHRASES (not in Kalshi list)":^60}')
print(f'{"="*60}')

# Show n-grams NOT already in the tracked terms list
for ngram, data in sorted(interesting_ngrams.items(), key=lambda x: x[1]['count'], reverse=True)[:30]:
    if ngram not in tracked_terms:
        print(f"{data['probability']:.1%} - \"{ngram}\" ({data['count']} total mentions)")

## 6. Export Predictions for Your Trading Bot

In [ ]:
from datetime import datetime

# Build the final predictions output with multi-scenario data
predictions_output = {
    'generated_at': datetime.utcnow().isoformat(),
    'event_context': EVENT_CONTEXT,
    'simulation_params': {
        'num_simulations': NUM_SIMULATIONS,
        'temperature': TEMPERATURE,
        'top_p': TOP_P,
        'max_tokens': MAX_NEW_TOKENS,
        'base_model': BASE_MODEL,
        'recency_half_life_days': HALF_LIFE_DAYS,
    },
    'scenario_weights_used': SCENARIO_WEIGHTS,
    'scenario_counts': dict(actual_counts),
    'gemini_enrichment': {
        'enriched_topics': enriched_topics,
        'scenario_hot_words': SCENARIO_HOT_WORDS,
    },
    'term_predictions': [
        {
            'term': term,
            'probability': data['probability'],
            'raw_probability': data['raw_probability'],
            'recency_weight': data['recency_weight'],
            'confidence': data['confidence'],
            'speeches_containing': data['speeches_containing'],
            'total_mentions': data['total_mentions'],
            'avg_mentions_per_speech': data['avg_mentions_per_speech'],
            'by_scenario': data['by_scenario'],
            'model_name': 'monte_carlo_multiscenario',
        }
        for term, data in sorted_terms
    ],
    'discovered_phrases': [
        {
            'phrase': ngram,
            'probability': data['probability'],
            'total_mentions': data['count'],
        }
        for ngram, data in sorted(interesting_ngrams.items(),
                                   key=lambda x: x[1]['count'], reverse=True)[:50]
    ],
}

# Save to Google Drive
timestamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
output_path = os.path.join(OUTPUT_DIR, f'predictions_{timestamp}.json')
latest_path = os.path.join(OUTPUT_DIR, 'predictions_latest.json')

for path in [output_path, latest_path]:
    with open(path, 'w') as f:
        json.dump(predictions_output, f, indent=2)

print(f'Predictions saved to:')
print(f'  {output_path}')
print(f'  {latest_path}')
print(f'\n📊 Summary:')
print(f'  Terms predicted: {len(predictions_output["term_predictions"])}')
print(f'  Discovered phrases: {len(predictions_output["discovered_phrases"])}')
print(f'  Scenarios: {list(predictions_output["scenario_counts"].keys())}')
print(f'\nDownload predictions_latest.json and place it in your bot\'s data/ directory.')
print(f'Or use the Colab API server (next cell) for live access.')

## 7. (Optional) Expose as API for Live Bot Access

Run a FastAPI server in Colab with ngrok tunnel so your local bot can call it.

In [ ]:
# Optional: Run as API server
RUN_API = False  # Set to True to expose the model as an API

if RUN_API:
    !pip install -q fastapi uvicorn pyngrok
    
    from fastapi import FastAPI
    from pydantic import BaseModel
    from pyngrok import ngrok
    import uvicorn
    import threading
    
    api = FastAPI(title='Trump Prediction API')
    
    class PredictionRequest(BaseModel):
        event_type: str = 'rally'
        location: str = 'unknown'
        audience: str = 'supporters'
        current_news: list[str] = []
        num_simulations: int = 100  # fewer for live API
        terms: list[str] = []
    
    @api.post('/predict')
    def predict(req: PredictionRequest):
        # Quick Monte Carlo run with fewer simulations
        prompt = build_generation_prompt(req.dict())
        speeches = []
        
        for _ in range(req.num_simulations):
            inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
            with torch.no_grad():
                output = model.generate(
                    **inputs, max_new_tokens=256,
                    temperature=0.75, do_sample=True,
                )
            speeches.append(tokenizer.decode(output[0], skip_special_tokens=True))
        
        results = {}
        terms_to_check = req.terms or tracked_terms
        for term in terms_to_check:
            containing = sum(1 for s in speeches if check_term_in_speech(s, term))
            results[term] = round(containing / len(speeches), 4)
        
        return {'predictions': results, 'num_simulations': len(speeches)}
    
    @api.get('/health')
    def health():
        return {'status': 'ok', 'model': BASE_MODEL}
    
    # Start ngrok tunnel
    # You need to set your ngrok auth token:
    # ngrok.set_auth_token('YOUR_NGROK_TOKEN')
    
    public_url = ngrok.connect(8080)
    print(f'\n{"="*60}')
    print(f'API available at: {public_url}')
    print(f'Set this in your bot\'s .env: COLAB_PREDICTION_URL={public_url}')
    print(f'{"="*60}\n')
    
    # Run in background thread
    thread = threading.Thread(
        target=uvicorn.run,
        args=(api,),
        kwargs={'host': '0.0.0.0', 'port': 8080},
        daemon=True,
    )
    thread.start()
    print('API server running in background. Keep this notebook open.')

## 6. Signal Completion (Auto-Pipeline)

Write the completion signal so your local server imports the predictions.

In [ ]:
if AUTO_TRIGGERED:
    # Write completion signal with prediction summary
    completion_data = {
        'completed_at': datetime.utcnow().isoformat(),
        'status': 'success',
        'phase': 'monte_carlo_predictions',
        'predictions_path': os.path.join(PREDICTIONS_DIR, 'predictions_latest.json'),
        'trigger_data': trigger_data,
    }

    # Include prediction count if available
    pred_path = os.path.join(PREDICTIONS_DIR, 'predictions_latest.json')
    if os.path.exists(pred_path):
        with open(pred_path) as f:
            pdata = json.load(f)
        completion_data['predictions_count'] = len(pdata.get('term_predictions', []))
        completion_data['discovered_phrases'] = len(pdata.get('discovered_phrases', []))

    os.makedirs(TRIGGER_DIR, exist_ok=True)
    completion_path = os.path.join(TRIGGER_DIR, 'training_complete.json')
    with open(completion_path, 'w') as f:
        json.dump(completion_data, f, indent=2)

    # Clean up trigger file
    if os.path.exists(TRIGGER_PATH):
        os.remove(TRIGGER_PATH)

    print(f"✅ Completion signal written to: {completion_path}")
    print(f"   Predictions: {completion_data.get('predictions_count', '?')} terms")
    print(f"   Discovered phrases: {completion_data.get('discovered_phrases', '?')}")
    print(f"   Your local server will import these on its next poll cycle.")
else:
    print("ℹ️  Manual run complete.")
    print("   Download predictions_latest.json from Drive, or use:")
    print("   POST /api/drive/download-predictions on your local server.")